# torch_diagnostics

Generated with **Lispyn** — originally created by **Eduardo J. Barrios**.

Source: [Lispyn](https://github.com/edujbarrios/lispyn) (Apache-2.0).

In [ ]:
import hy


# Inspect, debug and evaluate a PyTorch model from Hy
Build with lispyn build examples/torch_diagnostics.hy or run it directly.

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='torch_diagnostics', assignments=[['model-footprint', 'model-footprint']], prefix='')
import torch
torch.manual_seed(7)
model = torch.nn.Sequential(torch.nn.Linear(2, 4), torch.nn.ReLU(), torch.nn.Linear(4, 2))
x = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
y = torch.tensor([0, 1, 0, 1])
loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x, y), batch_size=2)
import lispyn.torch_diagnostics
print('Nominal model footprint:', lispyn.torch_diagnostics.model_footprint(model))


## Check backward gradients before updating parameters

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='torch_diagnostics', assignments=[['gradient-report', 'gradient-report'], ['assert-gradients', 'assert-gradients'], ['with-anomaly-detection', 'with-anomaly-detection']], prefix='')
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
import torch
_hy_anon_1 = None
with torch.autograd.detect_anomaly():
    logits = model(x)
    loss = torch.nn.CrossEntropyLoss()(logits, y)
    _hy_anon_1 = loss.backward()
import lispyn.torch_diagnostics
gradients = lispyn.torch_diagnostics.gradient_report(model)
assert gradients['all_finite']
import lispyn.torch_diagnostics
lispyn.torch_diagnostics.assert_gradients(model)
optimizer.step()
optimizer.zero_grad()
print('Gradient elements:', gradients['gradient_elements'])


## Temporarily freeze parameters and restore the deterministic policy

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='torch_diagnostics', assignments=[['with-parameter-freeze', 'with-parameter-freeze'], ['with-determinism', 'with-determinism']], prefix='')
first_parameter = next(iter(model.parameters()))
assert first_parameter.requires_grad
import lispyn.torch_diagnostics
_hy_anon_1 = None
with lispyn.torch_diagnostics.temporarily_freeze(model):
    assert not first_parameter.requires_grad
    _hy_anon_1 = None
assert first_parameter.requires_grad
import lispyn.torch_diagnostics
_hy_anon_2 = None
with lispyn.torch_diagnostics.deterministic_algorithms():
    assert torch.are_deterministic_algorithms_enabled()
    _hy_anon_2 = None
print('Trainable flags and policy restored')


## Collect logits from a loader without training

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='torch_diagnostics', assignments=[['predict-batches', 'predict-batches']], prefix='')
import lispyn.torch_diagnostics
predictions = lispyn.torch_diagnostics.predict_batches(model, loader, device='cpu', max_batches=2)
assert list(predictions.shape) == [4, 2]
assert model.training
print('Batched inference:', list(predictions.shape))
